In [2]:
customers = spark.read.csv("data/customers.csv", header=True, inferSchema=True)
products = spark.read.csv("data/products.csv", header=True, inferSchema=True)
orders = spark.read.csv("data/orders.csv", header=True, inferSchema=True)
order_items = spark.read.csv("data/order_items.csv", header=True, inferSchema=True)
returns = spark.read.csv("data/returns.csv", header=True, inferSchema=True)

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Case Study") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/16 05:31:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/16 05:31:20 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [3]:
customers = spark.read.csv("data/customers.csv", header=True, inferSchema=True)
products = spark.read.csv("data/products.csv", header=True, inferSchema=True)
orders = spark.read.csv("data/orders.csv", header=True, inferSchema=True)
order_items = spark.read.csv("data/order_items.csv", header=True, inferSchema=True)
returns = spark.read.csv("data/returns.csv", header=True, inferSchema=True)

In [4]:
print("Customers:", customers.count())
print("Products:", products.count())
print("Orders:", orders.count())
print("Returned Orders:", returns.select("order_id").distinct().count())
print("order_items:",order_items.count())

Customers: 100000
Products: 50000
Orders: 1000000


Returned Orders: 100000


[Stage 35:>                                                         (0 + 2) / 2]

order_items: 3000000


In [5]:
from pyspark.sql.functions import sum, col

sales = order_items.join(products, "product_id")

sales.groupBy("category") \
     .agg(sum(col("quantity") * col("unit_price"))
     .alias("total_sales")) \
     .show()

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `unit_price` cannot be resolved. Did you mean one of the following? [`unit_cost`, `selling_price`, `category`, `order_id`, `brand`].;
'Aggregate [category#183], [category#183, sum((quantity#238 * 'unit_price)) AS total_sales#341]
+- Project [product_id#237, order_item_id#235, order_id#236, quantity#238, selling_price#239, product_name#182, category#183, brand#184, unit_cost#185]
   +- Join Inner, (product_id#237 = product_id#181)
      :- Relation [order_item_id#235,order_id#236,product_id#237,quantity#238,selling_price#239] csv
      +- Relation [product_id#181,product_name#182,category#183,brand#184,unit_cost#185] csv


In [10]:
customers.printSchema()
products.printSchema()
orders.printSchema()
order_items.printSchema()
returns.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- customer_segment: string (nullable = true)

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- unit_cost: double (nullable = true)

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- order_status: string (nullable = true)

root
 |-- order_item_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- selling_price: double (nullable = true)

root
 |-- return_id: integer (nullable = true)
 |-- order_id: integer (nullable = tru

In [6]:
from pyspark.sql.functions import sum, col

sales = order_items.join(products, "product_id")

sales.groupBy("category") \
     .agg(
         sum(col("quantity") * col("selling_price"))
         .alias("total_sales")
     ) \
     .show(truncate=False)

[Stage 39:>                                                         (0 + 2) / 2]

+--------------+-------------------+
|category      |total_sales        |
+--------------+-------------------+
|Home & Kitchen|7.581388732799902E8|
|Sports        |7.433388681300008E8|
|Electronics   |7.442665041099958E8|
|Clothing      |7.419227945699946E8|
|Books         |7.464907783499908E8|
|Beauty        |7.626693058999963E8|
|Toys          |7.446190722999846E8|
+--------------+-------------------+



In [7]:
customer_sales = customers \
    .join(orders, "customer_id") \
    .join(order_items, "order_id")

customer_sales.groupBy(
    "customer_id",
    "customer_name"
).agg(
    sum(col("quantity") * col("selling_price"))
    .alias("total_purchase")
).orderBy(
    col("total_purchase").desc()
).show(10)

[Stage 47:=============================>                            (1 + 1) / 2]

+-----------+--------------+------------------+
|customer_id| customer_name|    total_purchase|
+-----------+--------------+------------------+
|      93094|Customer_93094|         181569.68|
|      64560|Customer_64560|169060.39999999997|
|      23289|Customer_23289|161573.80000000002|
|      52275|Customer_52275|153364.78999999998|
|      61218|Customer_61218|         153067.55|
|      52034|Customer_52034|         152680.05|
|      40442|Customer_40442|         151037.32|
|      60528|Customer_60528|         148691.95|
|      84830|Customer_84830|         148363.84|
|      82593|Customer_82593|148281.03999999998|
+-----------+--------------+------------------+
only showing top 10 rows



In [9]:
from pyspark.sql.functions import month, year

monthly_sales = orders \
    .join(order_items, "order_id")

monthly_sales.groupBy(
    year("order_date").alias("year"),
    month("order_date").alias("month")
).agg(       
    sum(col("quantity") * col("selling_price"))
    .alias("sales")
).orderBy(
    "year",
    "month"
).show() 

[Stage 56:=============================>                            (1 + 1) / 2]

+----+-----+--------------------+
|year|month|               sales|
+----+-----+--------------------+
|2024|    1| 4.445777757600032E8|
|2024|    2| 4.153661441999996E8|
|2024|    3| 4.436282454099993E8|
|2024|    4|4.2782097433999574E8|
|2024|    5|4.4481061894999886E8|
|2024|    6| 4.317051540600023E8|
|2024|    7| 4.436705191199994E8|
|2024|    8|4.4109517702000153E8|
|2024|    9| 4.310715260799986E8|
|2024|   10|4.4136378931000113E8|
|2024|   11| 4.336233640400014E8|
|2024|   12| 4.427129083499967E8|
+----+-----+--------------------+



In [10]:
total_sales = order_items.join(products, "product_id")

returned_sales = returns \
    .join(order_items, "order_id") \
    .join(products, "product_id")

total_count = total_sales.groupBy("category").count()

return_count = returned_sales.groupBy("category") \
                             .count() \
                             .withColumnRenamed("count", "returned")

result = total_count.join(
    return_count,
    "category",
    "left"
).fillna(0)

result = result.withColumn(
    "return_percentage",
    (col("returned") / col("count")) * 100
)

result.show() 

[Stage 65:>                                                         (0 + 2) / 2]

+--------------+------+--------+------------------+
|      category| count|returned| return_percentage|
+--------------+------+--------+------------------+
|Home & Kitchen|434034|   43418|10.003363791776682|
|        Sports|424412|   42530|10.020923065323316|
|   Electronics|425896|   42601|10.002676709807089|
|      Clothing|427607|   42660| 9.976450338745623|
|         Books|427086|   42809|10.023508145900358|
|        Beauty|430547|   43194| 10.03235419129619|
|          Toys|430418|   43382|10.079039445376354|
+--------------+------+--------+------------------+



In [15]:
from pyspark.sql.functions import count, row_number
from pyspark.sql.window import Window

payment_data = customers.join(
    orders,
    "customer_id"
)

payment_count = payment_data.groupBy(
    "state",
    "payment_mode"
).agg(
    count("*").alias("cnt")
)

window_spec = Window.partitionBy(
    "state"
).orderBy(
    col("cnt").desc()
)

payment_count.withColumn(
    "rank",
    row_number().over(window_spec)
).filter(
    col("rank")==1
).show() 

[Stage 85:>                                                         (0 + 2) / 2]

+-----+----------------+-----+----+
|state|    payment_mode|  cnt|rank|
+-----+----------------+-----+----+
|   CA|             UPI|20246|   1|
|   FL|      Debit Card|20010|   1|
|   GA|     Net Banking|20041|   1|
|   IL|Cash on Delivery|20498|   1|
|   MI|      Debit Card|20416|   1|
|   NC|     Net Banking|19596|   1|
|   NY|      Debit Card|20369|   1|
|   OH|     Net Banking|20351|   1|
|   TX|             UPI|20065|   1|
|   WA|             UPI|20244|   1|
+-----+----------------+-----+----+

